# Churn Risk Score & Churn Radar — Customer Retention Intelligence

## Project Overview

โปรเจกต์นี้วิเคราะห์พฤติกรรมการซื้อของลูกค้าเพื่อ:

1. **Churn Risk Score** — คะแนนความเสี่ยง 0-100 ว่าลูกค้าแต่ละคนจะหยุดซื้อในงวดถัดไป
2. **Churn Radar** — ระบบตรวจจับสัญญาณเตือนล่วงหน้า (Early Warning Signals)
3. **Cluster Pattern** — จัดกลุ่มรูปแบบพฤติกรรมของลูกค้าที่จะ Churn

---

## Project Plan

| Phase | Description | Models / Techniques |
|-------|------------|---------------------|
| **1. Data Loading & EDA** | โหลดข้อมูล 5 งวด, สำรวจ distribution, churn rate trend | Descriptive Statistics |
| **2. Feature Engineering** | สร้าง features จาก time-series ซื้อ: RFM, Trend, Decay, Dormancy | Domain Knowledge |
| **3. Statistical Testing** | ทดสอบว่า features ใดแยก Churn vs Retain ได้ | Mann-Whitney U, Chi-Square, Effect Size |
| **4. ML Models** | สร้างโมเดลทำนาย Churn | Logistic Regression, Random Forest, XGBoost, LightGBM |
| **5. Risk Score** | Calibrated probability → Risk Score 0-100 | Platt Scaling, Isotonic Regression |
| **6. Survival Analysis** | วิเคราะห์ time-to-churn | Kaplan-Meier, Cox Proportional Hazards |
| **7. Churn Clustering** | จัดกลุ่มรูปแบบ Churn | K-Means, PCA, Cluster Profiling |
| **8. Churn Radar Dashboard** | Cohort analysis, trend monitoring, early warning | Visualization |
| **9. Production Pipeline** | Reusable code สำหรับ data งวดถัดไป | Pipeline Class |

---

## Phase 1: Data Loading & EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# --- Load all 5 datasets ---
DATA_DIR = Path('.')
FILES = sorted(DATA_DIR.glob('Churn_*.csv'))

datasets = {}
for f in FILES:
    key = f.stem  # e.g. Churn_1216_0102
    datasets[key] = pd.read_csv(f, low_memory=False)
    print(f"{key}: {datasets[key].shape[0]:,} rows x {datasets[key].shape[1]} cols | churn_rate={1 - datasets[key]['label_continue'].mean():.2%}")

print(f"\nTotal files: {len(datasets)}")

In [ ]:
# --- Churn rate trend across periods ---
period_labels = []
churn_rates = []
total_customers = []
for key in sorted(datasets.keys()):
    df = datasets[key]
    period_labels.append(key.replace('Churn_', ''))
    churn_rates.append(1 - df['label_continue'].mean())
    total_customers.append(len(df))

fig, ax1 = plt.subplots(figsize=(12, 5))
color1, color2 = '#e74c3c', '#3498db'

ax1.bar(period_labels, total_customers, color=color2, alpha=0.3, label='Total Customers')
ax1.set_ylabel('Total Customers', color=color2)
ax1.tick_params(axis='y', labelcolor=color2)

ax2 = ax1.twinx()
ax2.plot(period_labels, churn_rates, color=color1, marker='o', linewidth=2.5, markersize=10, label='Churn Rate')
ax2.set_ylabel('Churn Rate', color=color1)
ax2.tick_params(axis='y', labelcolor=color1)
for i, r in enumerate(churn_rates):
    ax2.annotate(f'{r:.1%}', (period_labels[i], r), textcoords="offset points", xytext=(0, 12), ha='center', fontweight='bold', color=color1)

plt.title('Churn Rate & Customer Base Trend Across Periods')
fig.tight_layout()
plt.show()

In [ ]:
# --- EDA: Demographics distribution (latest period) ---
df_latest = datasets[sorted(datasets.keys())[-1]].copy()
df_latest['churn'] = 1 - df_latest['label_continue']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
axes[0].hist(df_latest[df_latest['churn']==0]['age'].dropna(), bins=40, alpha=0.6, label='Retain', color='#3498db', density=True)
axes[0].hist(df_latest[df_latest['churn']==1]['age'].dropna(), bins=40, alpha=0.6, label='Churn', color='#e74c3c', density=True)
axes[0].set_title('Age Distribution by Churn')
axes[0].set_xlabel('Age')
axes[0].legend()

# Gender
gender_churn = df_latest.groupby('gender')['churn'].agg(['mean', 'count']).reset_index()
gender_churn.columns = ['gender', 'churn_rate', 'count']
bars = axes[1].bar(gender_churn['gender'], gender_churn['churn_rate'], color=['#3498db', '#e74c3c', '#95a5a6'])
axes[1].set_title('Churn Rate by Gender')
axes[1].set_ylabel('Churn Rate')
for bar, rate in zip(bars, gender_churn['churn_rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{rate:.1%}', ha='center', fontweight='bold')

# Top 15 provinces by churn rate (min 1000 customers)
prov_churn = df_latest.groupby('province')['churn'].agg(['mean', 'count']).reset_index()
prov_churn.columns = ['province', 'churn_rate', 'count']
prov_churn = prov_churn[prov_churn['count'] >= 1000].nlargest(15, 'churn_rate')
axes[2].barh(prov_churn['province'], prov_churn['churn_rate'], color='#e74c3c', alpha=0.7)
axes[2].set_title('Top 15 Provinces by Churn Rate (n>=1000)')
axes[2].set_xlabel('Churn Rate')

plt.tight_layout()
plt.show()

## Phase 2: Feature Engineering

สร้าง features จากข้อมูลการซื้อแบบ time-series โดยแบ่งเป็น **13 กลุ่ม (35+ features)**:

| Feature Group | Features | Rationale |
|--------------|----------|-----------|
| **Recency** | rounds_since_last_purchase | ยิ่งนานไม่ซื้อ ยิ่งเสี่ยง |
| **Frequency** | total_active_rounds, purchase_frequency_ratio | ซื้อบ่อยแค่ไหน |
| **Volume** | total_items, avg_items_per_active_round, max_single_round | ปริมาณซื้อ |
| **Trend** | trend_slope, recent_vs_early_ratio | แนวโน้มเพิ่ม/ลด |
| **Momentum** | ewm_recent, purchase_acceleration | ความเร่งของพฤติกรรม |
| **Consistency** | purchase_std, coeff_of_variation | ความสม่ำเสมอ |
| **Dormancy** | longest_zero_streak, current_zero_streak | ช่วงหยุดซื้อ |
| **Lifecycle** | tenure_rounds, time_to_first_purchase | อายุลูกค้า |
| **Recent Window** | items_last_1/3/6_rounds, active_last_3/6_rounds | พฤติกรรมล่าสุด |
| **Inter-purchase Interval** | avg_gap, std_gap, max_gap, recent_gap_vs_avg | จังหวะ/rhythm การซื้อ |
| **Reactivation** | n_reactivations, ever_reactivated | เคยหยุดแล้วกลับมากี่ครั้ง |
| **Volume Trend (granular)** | items_last3_vs_prev3_ratio, is_declining_3_consecutive | ซื้อลดลงช่วงล่าสุดหรือไม่ |
| **Spending Concentration** | gini_coefficient, top3_rounds_pct | กระจายหรือกระจุก |
| **Tenure Stage** | is_new, is_mature | ลูกค้าใหม่ vs เก่า |

In [ ]:
def get_item_columns(df):
    """Get item columns in chronological order."""
    return [c for c in df.columns if c.startswith('item')]

def build_purchase_matrix(df):
    """Convert item columns to numeric matrix. null -> NaN (not registered), else int."""
    item_cols = get_item_columns(df)
    mat = df[item_cols].replace('null', np.nan).astype(float)
    return mat, item_cols

def engineer_features(df):
    """Build all features from a single period's dataframe. (Optimized — vectorized)"""
    mat, item_cols = build_purchase_matrix(df)
    n_rounds = len(item_cols)
    
    mat_vals = mat.values  # numpy array for speed
    registered = ~np.isnan(mat_vals)
    purchased = np.where(registered, mat_vals > 0, False)
    amounts = np.where(registered, mat_vals, 0.0)
    
    feats = pd.DataFrame(index=df.index)
    
    # ============================
    # --- Lifecycle (vectorized) ---
    # ============================
    has_any_reg = registered.any(axis=1)
    first_reg_idx = np.where(has_any_reg, registered.argmax(axis=1), n_rounds)
    feats['tenure_rounds'] = np.where(has_any_reg, n_rounds - first_reg_idx, 0)
    
    has_any_purchase = purchased.any(axis=1)
    first_purchase_idx = np.where(has_any_purchase, purchased.argmax(axis=1), n_rounds)
    feats['time_to_first_purchase'] = np.where(
        has_any_purchase, first_purchase_idx - first_reg_idx, feats['tenure_rounds'])
    
    # ============================
    # --- Recency (vectorized) ---
    # ============================
    rev_purchased = np.flip(purchased, axis=1)
    last_purchase_idx = np.where(has_any_purchase,
                                  n_rounds - 1 - rev_purchased.argmax(axis=1), -1)
    feats['rounds_since_last_purchase'] = np.where(
        has_any_purchase, (n_rounds - 1) - last_purchase_idx, feats['tenure_rounds'])
    
    # ============================
    # --- Frequency (vectorized) ---
    # ============================
    feats['total_active_rounds'] = purchased.sum(axis=1)
    registered_rounds = np.maximum(registered.sum(axis=1), 1)
    feats['purchase_frequency_ratio'] = feats['total_active_rounds'] / registered_rounds
    
    # ============================
    # --- Volume (vectorized) ---
    # ============================
    feats['total_items'] = amounts.sum(axis=1)
    feats['avg_items_per_active_round'] = np.where(
        feats['total_active_rounds'] > 0,
        feats['total_items'] / feats['total_active_rounds'], 0)
    feats['max_single_round'] = amounts.max(axis=1)
    
    # ============================
    # --- Recent Windows (vectorized) ---
    # ============================
    feats['items_last_1_round'] = amounts[:, -1]
    feats['items_last_3_rounds'] = amounts[:, -3:].sum(axis=1)
    feats['items_last_6_rounds'] = amounts[:, -6:].sum(axis=1)
    feats['active_last_3_rounds'] = purchased[:, -3:].sum(axis=1)
    feats['active_last_6_rounds'] = purchased[:, -6:].sum(axis=1)
    
    # ============================
    # --- Trend (vectorized linear regression) ---
    # ============================
    # Compute slope per row using vectorized formula over registered rounds
    # slope = (n*sum(xy) - sum(x)*sum(y)) / (n*sum(x^2) - (sum(x))^2)
    x_grid = np.arange(n_rounds, dtype=float)
    # Use only registered values: mask out unregistered
    x_masked = np.where(registered, x_grid[np.newaxis, :], 0.0)
    y_masked = amounts.copy()
    n_valid = registered.sum(axis=1).astype(float)
    
    sum_x = (x_masked * registered).sum(axis=1)
    sum_y = y_masked.sum(axis=1)
    sum_xy = (x_masked * y_masked).sum(axis=1)
    sum_x2 = (x_masked ** 2 * registered).sum(axis=1)
    
    denom = n_valid * sum_x2 - sum_x ** 2
    slope = np.where((denom != 0) & (n_valid >= 3),
                     (n_valid * sum_xy - sum_x * sum_y) / denom, 0.0)
    # Set slope to 0 if y has zero variance
    y_mean = np.where(n_valid > 0, sum_y / n_valid, 0.0)
    y_var = np.where(n_valid > 0,
                     (np.where(registered, (amounts - y_mean[:, np.newaxis]) ** 2, 0.0)).sum(axis=1) / n_valid,
                     0.0)
    feats['trend_slope'] = np.where(y_var > 0, slope, 0.0)
    
    half = max(1, n_rounds // 2)
    recent_sum = amounts[:, -half:].sum(axis=1)
    early_sum = amounts[:, :half].sum(axis=1)
    feats['recent_vs_early_ratio'] = np.where(
        early_sum > 0, recent_sum / early_sum,
        np.where(recent_sum > 0, 2.0, 0.0))
    
    # ============================
    # --- Momentum (vectorized) ---
    # ============================
    amt_df = pd.DataFrame(amounts)
    feats['ewm_recent'] = amt_df.T.ewm(span=5, adjust=False).mean().iloc[-1].values
    
    roll3 = amt_df.T.rolling(3, min_periods=1).mean().T
    if roll3.shape[1] >= 6:
        feats['purchase_acceleration'] = roll3.iloc[:, -1].values - roll3.iloc[:, -4].values
    else:
        feats['purchase_acceleration'] = 0.0
    
    # ============================
    # --- Consistency (vectorized) ---
    # ============================
    feats['purchase_std'] = amounts.std(axis=1)
    mean_amt = amounts.mean(axis=1)
    feats['coeff_of_variation'] = np.where(mean_amt > 0, amounts.std(axis=1) / mean_amt, 0.0)
    
    # ============================
    # --- Dormancy: current_zero_streak (vectorized) ---
    # ============================
    # Since NaN is at the beginning, trailing zeros from end = trailing zeros in valid portion
    rev_amounts = np.flip(amounts, axis=1)
    rev_nonzero = rev_amounts > 0
    has_any_nonzero = rev_nonzero.any(axis=1)
    first_nonzero_from_end = rev_nonzero.argmax(axis=1)
    feats['current_zero_streak'] = np.where(
        has_any_nonzero, first_nonzero_from_end,
        np.where(has_any_reg, registered.sum(axis=1), 0))
    
    # ============================
    # --- Dormancy: longest_zero_streak + gaps + reactivations (single pass) ---
    # --- Spending Concentration: gini + top3 (vectorized) ---
    # ============================
    # Combine all remaining row-wise ops into ONE apply for efficiency
    def compute_row_features(i):
        row = mat_vals[i]
        valid = row[~np.isnan(row)]
        nv = len(valid)
        
        # longest_zero_streak
        if nv == 0:
            return (0, 0.0, 0.0, 0.0, 0.0, 0, 0)
        max_streak = 0; cur = 0
        for v in valid:
            if v == 0: cur += 1; max_streak = max(max_streak, cur)
            else: cur = 0
        
        # gaps between purchases
        purchase_idx = np.where(valid > 0)[0]
        if len(purchase_idx) >= 2:
            gaps = np.diff(purchase_idx)
            avg_gap = gaps.mean()
            std_gap = gaps.std() if len(gaps) > 1 else 0.0
            max_gap = float(gaps.max())
            recent_gap_ratio = gaps[-1] / avg_gap if avg_gap > 0 else 0.0
        else:
            avg_gap = std_gap = max_gap = recent_gap_ratio = 0.0
        
        # reactivations
        if nv >= 3:
            is_act = (valid > 0)
            reacts = 0; was_active = False
            for j in range(nv - 1):
                if is_act[j]: was_active = True
                if was_active and not is_act[j] and is_act[j + 1]: reacts += 1
        else:
            reacts = 0
        
        return (max_streak, avg_gap, std_gap, max_gap, recent_gap_ratio, reacts, 1 if reacts > 0 else 0)
    
    print("    Computing row-wise features (streak, gaps, reactivations)...")
    n_rows = len(mat_vals)
    row_results = np.zeros((n_rows, 7), dtype=float)
    for i in range(n_rows):
        row_results[i] = compute_row_features(i)
    
    feats['longest_zero_streak'] = row_results[:, 0]
    feats['avg_gap'] = row_results[:, 1]
    feats['std_gap'] = row_results[:, 2]
    feats['max_gap'] = row_results[:, 3]
    feats['recent_gap_vs_avg'] = row_results[:, 4]
    feats['n_reactivations'] = row_results[:, 5].astype(int)
    feats['ever_reactivated'] = row_results[:, 6].astype(int)
    
    # ============================
    # --- Volume Trend granular (vectorized) ---
    # ============================
    if n_rounds >= 6:
        items_last3 = amounts[:, -3:].sum(axis=1)
        items_prev3 = amounts[:, -6:-3].sum(axis=1)
        feats['items_last3_vs_prev3_ratio'] = np.where(
            items_prev3 > 0, items_last3 / items_prev3,
            np.where(items_last3 > 0, 2.0, 0.0))
        feats['is_declining_3_consecutive'] = (
            (amounts[:, -1] <= amounts[:, -2]) &
            (amounts[:, -2] <= amounts[:, -3]) &
            (amounts[:, -3] > 0)).astype(int)
    else:
        feats['items_last3_vs_prev3_ratio'] = 0.0
        feats['is_declining_3_consecutive'] = 0
    
    # ============================
    # --- Spending Concentration (vectorized) ---
    # ============================
    sorted_amounts = np.sort(amounts, axis=1)
    n_cols = sorted_amounts.shape[1]
    index_arr = np.arange(1, n_cols + 1)
    total = sorted_amounts.sum(axis=1)
    weighted_sum = (sorted_amounts * index_arr[np.newaxis, :]).sum(axis=1)
    feats['gini_coefficient'] = np.where(
        total > 0, (2 * weighted_sum - (n_cols + 1) * total) / (n_cols * total), 0.0)
    
    top3_sum = sorted_amounts[:, -3:].sum(axis=1)
    feats['top3_rounds_pct'] = np.where(total > 0, top3_sum / total, 0.0)
    
    # ============================
    # --- Tenure Stage (vectorized) ---
    # ============================
    feats['is_new'] = (feats['tenure_rounds'] <= 3).astype(int)
    feats['is_mature'] = (feats['tenure_rounds'] >= 15).astype(int)
    
    # ============================
    # --- Demographics ---
    # ============================
    feats['age'] = df['age'].values
    feats['gender_encoded'] = df['gender'].map({'MALE': 0, 'FEMALE': 1}).fillna(2).astype(int).values
    
    return feats

print("Feature engineering function defined (v3 — vectorized)")
print("Optimizations: trend_slope, gini, top3, current_zero_streak fully vectorized")
print("Remaining row-ops combined into single pass (streak + gaps + reactivations)")

In [ ]:
%%time
# Process all datasets and stack them for training
all_features = []
all_labels = []
all_periods = []
all_user_nos = []

for key in sorted(datasets.keys()):
    print(f"Processing {key}...")
    df = datasets[key]
    feats = engineer_features(df)
    feats['period'] = key
    all_features.append(feats)
    all_labels.append(df['label_continue'].values)
    all_periods.append([key] * len(df))
    all_user_nos.append(df['userNo'].values)

features_df = pd.concat(all_features, ignore_index=True)
labels = np.concatenate(all_labels)
periods = np.concatenate(all_periods)
user_nos = np.concatenate(all_user_nos)

features_df['label'] = labels
features_df['churn'] = 1 - labels
features_df['userNo'] = user_nos

print(f"\nCombined dataset: {features_df.shape}")
features_df.head()

In [ ]:
# --- Feature distributions: Churn vs Retain ---
feature_cols = [c for c in features_df.columns if c not in ['period', 'label', 'churn', 'userNo']]
print(f"Total features: {len(feature_cols)}")

n_feats = len(feature_cols)
n_cols_grid = 6
n_rows_grid = (n_feats + n_cols_grid - 1) // n_cols_grid

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(28, 4 * n_rows_grid))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    data_retain = features_df.loc[features_df['churn']==0, col].dropna()
    data_churn = features_df.loc[features_df['churn']==1, col].dropna()
    
    p99 = features_df[col].quantile(0.99)
    ax.hist(data_retain.clip(upper=p99), bins=40, alpha=0.5, label='Retain', color='#3498db', density=True)
    ax.hist(data_churn.clip(upper=p99), bins=40, alpha=0.5, label='Churn', color='#e74c3c', density=True)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=6)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Churn vs Retain (All 35+ Features)', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## Phase 3: Statistical Testing

ทดสอบทางสถิติว่า feature แต่ละตัวแยก Churn vs Retain ได้อย่างมีนัยสำคัญหรือไม่
- **Mann-Whitney U Test** สำหรับ numerical features (non-parametric, ไม่ต้อง assume normal distribution)
- **Effect Size (Cohen's d)** เพื่อวัดขนาดความแตกต่าง

In [ ]:
# --- Statistical Tests ---
numeric_feats = [c for c in feature_cols if c != 'gender_encoded']

results = []
churn_mask = features_df['churn'] == 1
retain_mask = features_df['churn'] == 0

for col in numeric_feats:
    churn_vals = features_df.loc[churn_mask, col].dropna()
    retain_vals = features_df.loc[retain_mask, col].dropna()
    
    # Mann-Whitney U
    u_stat, p_val = stats.mannwhitneyu(churn_vals, retain_vals, alternative='two-sided')
    
    # Cohen's d
    pooled_std = np.sqrt((churn_vals.std()**2 + retain_vals.std()**2) / 2)
    cohens_d = (churn_vals.mean() - retain_vals.mean()) / pooled_std if pooled_std > 0 else 0
    
    results.append({
        'feature': col,
        'churn_mean': churn_vals.mean(),
        'retain_mean': retain_vals.mean(),
        'p_value': p_val,
        'cohens_d': cohens_d,
        'abs_cohens_d': abs(cohens_d),
        'effect_size': 'Large' if abs(cohens_d) >= 0.8 else ('Medium' if abs(cohens_d) >= 0.5 else ('Small' if abs(cohens_d) >= 0.2 else 'Negligible')),
        'significant': p_val < 0.001
    })

stat_results = pd.DataFrame(results).sort_values('abs_cohens_d', ascending=False)
print("Statistical Test Results (sorted by effect size):\n")
display(stat_results[['feature', 'churn_mean', 'retain_mean', 'cohens_d', 'effect_size', 'p_value', 'significant']].to_string(index=False))

# Visualize top features by effect size
fig, ax = plt.subplots(figsize=(12, 8))
top_feats = stat_results.head(15)
colors = ['#e74c3c' if d > 0 else '#3498db' for d in top_feats['cohens_d']]
ax.barh(top_feats['feature'], top_feats['cohens_d'], color=colors)
ax.set_xlabel("Cohen's d (positive = higher in Churn)")
ax.set_title("Top 15 Features by Effect Size (Churn vs Retain)")
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap of features ---
corr = features_df[numeric_feats + ['churn']].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Show features most correlated with churn
churn_corr = corr['churn'].drop('churn').abs().sort_values(ascending=False)
print("\nTop 10 features correlated with Churn:")
print(corr['churn'].drop('churn').reindex(churn_corr.index).head(10))

## Phase 4: ML Models — Churn Prediction

**Strategy:**
- **Train**: 4 งวดแรก (temporal split เพื่อป้องกัน data leakage)
- **Test**: งวดล่าสุด (จำลองการ deploy จริง)
- **Models**: Logistic Regression (baseline), Random Forest, XGBoost, LightGBM
- **Evaluation**: AUC-ROC, AUC-PR, F1, Precision, Recall + Calibration

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             classification_report, confusion_matrix, 
                             roc_curve, precision_recall_curve, brier_score_loss)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import xgboost as xgb
import lightgbm as lgb

# --- Temporal Train/Test Split ---
sorted_keys = sorted(datasets.keys())
last_period = sorted_keys[-1]

train_mask = features_df['period'] != last_period
test_mask = features_df['period'] == last_period

model_features = [c for c in feature_cols if c != 'gender_encoded'] + ['gender_encoded']

X_train = features_df.loc[train_mask, model_features].fillna(0)
y_train = features_df.loc[train_mask, 'churn'].astype(int)
X_test = features_df.loc[test_mask, model_features].fillna(0)
y_test = features_df.loc[test_mask, 'churn'].astype(int)

print(f"Train: {X_train.shape} | Churn rate: {y_train.mean():.2%}")
print(f"Test:  {X_test.shape} | Churn rate: {y_test.mean():.2%}")

# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
%%time
# --- Train all models ---
models = {}

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1, random_state=42)
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = ('scaled', lr)
print("1/4 Logistic Regression done")

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_leaf=50,
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['Random Forest'] = ('raw', rf)
print("2/4 Random Forest done")

# 3. XGBoost
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    scale_pos_weight=scale_pos, subsample=0.8, colsample_bytree=0.8,
    eval_metric='auc', random_state=42, n_jobs=-1,
    early_stopping_rounds=30
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
models['XGBoost'] = ('raw', xgb_model)
print("3/4 XGBoost done")

# 4. LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    scale_pos_weight=scale_pos, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
              callbacks=[lgb.early_stopping(30, verbose=False)])
models['LightGBM'] = ('raw', lgb_model)
print("4/4 LightGBM done")

In [ ]:
# --- Evaluate all models ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

eval_results = []

for name, (data_type, model) in models.items():
    X_eval = X_test_scaled if data_type == 'scaled' else X_test
    y_prob = model.predict_proba(X_eval)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    auc_roc = roc_auc_score(y_test, y_prob)
    auc_pr = average_precision_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    brier = brier_score_loss(y_test, y_prob)
    
    eval_results.append({
        'Model': name, 'AUC-ROC': auc_roc, 'AUC-PR': auc_pr,
        'F1': f1, 'Brier Score': brier
    })
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_roc:.4f})', linewidth=2)
    
    # PR Curve
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(rec, prec, label=f'{name} (AP={auc_pr:.4f})', linewidth=2)
    
    # Calibration Curve
    prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=15)
    axes[2].plot(prob_pred, prob_true, marker='o', label=name, linewidth=2)

axes[0].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].legend()

axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

axes[2].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[2].set_title('Calibration Curve')
axes[2].set_xlabel('Mean Predicted Probability')
axes[2].set_ylabel('Fraction of Positives')
axes[2].legend()

plt.tight_layout()
plt.show()

# Summary table
eval_df = pd.DataFrame(eval_results).sort_values('AUC-ROC', ascending=False)
print("\nModel Comparison:")
display(eval_df)

In [ ]:
# --- Feature Importance (Best Model) ---
# Use LightGBM feature importance + Logistic Regression coefficients side by side

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# LightGBM importance
lgb_imp = pd.Series(lgb_model.feature_importances_, index=model_features).sort_values(ascending=True)
lgb_imp.tail(15).plot(kind='barh', ax=axes[0], color='#2ecc71')
axes[0].set_title('LightGBM Feature Importance (top 15)')
axes[0].set_xlabel('Importance')

# Logistic Regression coefficients
lr_coefs = pd.Series(lr.coef_[0], index=model_features).sort_values()
top_bottom = pd.concat([lr_coefs.head(8), lr_coefs.tail(8)])
colors = ['#3498db' if v < 0 else '#e74c3c' for v in top_bottom]
top_bottom.plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_title('Logistic Regression Coefficients\n(Red=increases churn, Blue=decreases churn)')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

# Classification report for best model
best_name = eval_df.iloc[0]['Model']
best_data_type, best_model = models[best_name]
X_eval = X_test_scaled if best_data_type == 'scaled' else X_test
y_pred_best = (best_model.predict_proba(X_eval)[:, 1] >= 0.5).astype(int)
print(f"\nClassification Report — {best_name}:")
print(classification_report(y_test, y_pred_best, target_names=['Retain', 'Churn']))

## Phase 5: Risk Score (0-100)

แปลง probability ของ best model เป็น **Risk Score 0-100** ด้วย Isotonic Calibration เพื่อให้ probability สะท้อนความเป็นจริง

| Risk Level | Score | Action |
|-----------|-------|--------|
| Low | 0-25 | Monitor ปกติ |
| Medium | 26-50 | ส่ง promotion เบาๆ |
| High | 51-75 | ติดต่อเชิงรุก |
| Critical | 76-100 | Intervention ด่วน |

In [ ]:
# --- Calibrated Risk Score ---
# Use the best performing tree model and calibrate with isotonic regression
best_tree_name = eval_df[eval_df['Model'].isin(['XGBoost', 'LightGBM'])].iloc[0]['Model']
_, best_tree = models[best_tree_name]

# Rebuild model without early_stopping for CalibratedClassifierCV compatibility
if best_tree_name == 'XGBoost':
    base_for_cal = xgb.XGBClassifier(
        n_estimators=best_tree.best_iteration if hasattr(best_tree, 'best_iteration') and best_tree.best_iteration else 200,
        max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pos, subsample=0.8, colsample_bytree=0.8,
        eval_metric='auc', random_state=42, n_jobs=-1
    )
elif best_tree_name == 'LightGBM':
    base_for_cal = lgb.LGBMClassifier(
        n_estimators=best_tree.best_iteration_ if hasattr(best_tree, 'best_iteration_') and best_tree.best_iteration_ else 200,
        max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pos, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )

calibrated_model = CalibratedClassifierCV(base_for_cal, method='isotonic', cv=5)
calibrated_model.fit(X_train, y_train)

# Risk Score = calibrated probability * 100
risk_probs = calibrated_model.predict_proba(X_test)[:, 1]
risk_scores = (risk_probs * 100).round(1)

# Compare calibration before/after
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Before calibration
raw_probs = best_tree.predict_proba(X_test)[:, 1]
prob_true_raw, prob_pred_raw = calibration_curve(y_test, raw_probs, n_bins=15)
axes[0].plot(prob_pred_raw, prob_true_raw, 'o-', color='#e74c3c', label='Before Calibration')
prob_true_cal, prob_pred_cal = calibration_curve(y_test, risk_probs, n_bins=15)
axes[0].plot(prob_pred_cal, prob_true_cal, 's-', color='#2ecc71', label='After Calibration')
axes[0].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[0].set_title(f'{best_tree_name} Calibration')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Actual Fraction')
axes[0].legend()

# Risk Score Distribution
axes[1].hist(risk_scores[y_test==0], bins=50, alpha=0.6, label='Retain', color='#3498db', density=True)
axes[1].hist(risk_scores[y_test==1], bins=50, alpha=0.6, label='Churn', color='#e74c3c', density=True)
axes[1].set_title('Risk Score Distribution')
axes[1].set_xlabel('Risk Score (0-100)')
axes[1].legend()

# Risk Level breakdown
def risk_level(score):
    if score <= 25: return 'Low (0-25)'
    elif score <= 50: return 'Medium (26-50)'
    elif score <= 75: return 'High (51-75)'
    else: return 'Critical (76-100)'

risk_df = pd.DataFrame({'risk_score': risk_scores, 'actual_churn': y_test.values})
risk_df['risk_level'] = risk_df['risk_score'].apply(risk_level)

level_stats = risk_df.groupby('risk_level').agg(
    count=('actual_churn', 'size'),
    actual_churn_rate=('actual_churn', 'mean')
).sort_index()

bars = axes[2].bar(level_stats.index, level_stats['actual_churn_rate'], 
       color=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'])
for bar, rate, count in zip(bars, level_stats['actual_churn_rate'], level_stats['count']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{rate:.0%}\n(n={count:,})', ha='center', fontweight='bold', fontsize=9)
axes[2].set_title('Actual Churn Rate by Risk Level')
axes[2].set_ylabel('Actual Churn Rate')
axes[2].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print(f"\nBrier Score (calibrated): {brier_score_loss(y_test, risk_probs):.4f}")
print(f"AUC-ROC (calibrated): {roc_auc_score(y_test, risk_probs):.4f}")

## Phase 6: Survival Analysis

วิเคราะห์ว่าลูกค้ามี "อายุขัย" (time-to-churn) อย่างไร
- **Kaplan-Meier**: ดู survival curve รวมและแยกตามกลุ่ม
- **Cox Proportional Hazards**: หา hazard ratio ของแต่ละ feature

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFitter

# --- Build survival data from the latest period ---
# Duration = tenure_rounds (how long customer has been active)
# Event = churn (1 = churned, 0 = censored/still active)

surv_df = features_df[test_mask][['tenure_rounds', 'churn', 'age', 'gender_encoded',
                                    'total_active_rounds', 'purchase_frequency_ratio',
                                    'avg_items_per_active_round', 'current_zero_streak',
                                    'items_last_3_rounds', 'trend_slope', 'ewm_recent']].copy()
surv_df = surv_df[surv_df['tenure_rounds'] > 0].dropna()

# --- Kaplan-Meier Curve ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

kmf = KaplanMeierFitter()
kmf.fit(surv_df['tenure_rounds'], event_observed=surv_df['churn'], label='All Customers')
kmf.plot_survival_function(ax=axes[0], ci_show=True)
axes[0].set_title('Kaplan-Meier Survival Curve (All Customers)')
axes[0].set_xlabel('Tenure (Rounds)')
axes[0].set_ylabel('Survival Probability')

# KM by purchase frequency groups
surv_df['freq_group'] = pd.cut(surv_df['purchase_frequency_ratio'], 
                                bins=[0, 0.25, 0.5, 0.75, 1.01],
                                labels=['Very Low (<25%)', 'Low (25-50%)', 
                                        'Medium (50-75%)', 'High (>75%)'])

for group in surv_df['freq_group'].dropna().unique():
    mask = surv_df['freq_group'] == group
    kmf_g = KaplanMeierFitter()
    kmf_g.fit(surv_df.loc[mask, 'tenure_rounds'], 
              event_observed=surv_df.loc[mask, 'churn'], label=str(group))
    kmf_g.plot_survival_function(ax=axes[1], ci_show=False)

axes[1].set_title('Survival Curve by Purchase Frequency')
axes[1].set_xlabel('Tenure (Rounds)')
axes[1].set_ylabel('Survival Probability')
axes[1].legend(title='Frequency Group')

plt.tight_layout()
plt.show()

In [ ]:
# --- Cox Proportional Hazards Model ---
cox_features = ['tenure_rounds', 'age', 'gender_encoded', 'purchase_frequency_ratio',
                'avg_items_per_active_round', 'current_zero_streak',
                'items_last_3_rounds', 'trend_slope', 'ewm_recent']

cox_df = surv_df[cox_features + ['churn']].copy()
# Standardize features for Cox model
for col in cox_features:
    if col != 'tenure_rounds':
        cox_df[col] = (cox_df[col] - cox_df[col].mean()) / (cox_df[col].std() + 1e-8)

cph = CoxPHFitter(penalizer=0.01)
cph.fit(cox_df, duration_col='tenure_rounds', event_col='churn')

print("Cox Proportional Hazards — Summary")
cph.print_summary()

fig, ax = plt.subplots(figsize=(10, 6))
cph.plot(ax=ax)
ax.set_title('Cox PH — Hazard Ratios (exp(coef))\nHR > 1: increases churn risk, HR < 1: protective')
plt.tight_layout()
plt.show()

## Phase 7: Churn Clustering — Churn Archetypes

จัดกลุ่มลูกค้าที่ Churn เพื่อหา **รูปแบบการหลุด** ที่แตกต่างกัน → ช่วยวาง strategy ต่างกันสำหรับแต่ละกลุ่ม

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler as SS

# Focus on churned customers only
churn_feats = features_df[test_mask & (features_df['churn'] == 1)][model_features].fillna(0).copy()
churn_idx = churn_feats.index

# Scale for clustering
scaler_cl = SS()
churn_scaled = scaler_cl.fit_transform(churn_feats)

# --- Elbow Method ---
inertias = []
sil_scores = []
K_range = range(2, 9)

from sklearn.metrics import silhouette_score

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(churn_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(churn_scaled, km.labels_, sample_size=min(50000, len(churn_scaled))))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, sil_scores, 'ro-', linewidth=2)
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(sil_scores)]
print(f"Best K by silhouette: {best_k}")

In [ ]:
# --- Final Clustering with best K ---
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=20, max_iter=500)
cluster_labels = km_final.fit_predict(churn_scaled)

# PCA for visualization
pca = PCA(n_components=3)
pca_result = pca.fit_transform(churn_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 2D PCA
scatter = axes[0].scatter(pca_result[:, 0], pca_result[:, 1], c=cluster_labels, 
                          cmap='tab10', alpha=0.3, s=5)
axes[0].set_title(f'Churn Clusters (K={best_k}) — PCA 2D')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# 3D PCA
ax3d = fig.add_subplot(122, projection='3d')
axes[1].set_visible(False)
scatter3d = ax3d.scatter(pca_result[:, 0], pca_result[:, 1], pca_result[:, 2],
                         c=cluster_labels, cmap='tab10', alpha=0.3, s=3)
ax3d.set_title(f'Churn Clusters — PCA 3D')
ax3d.set_xlabel(f'PC1')
ax3d.set_ylabel(f'PC2')
ax3d.set_zlabel(f'PC3')

plt.tight_layout()
plt.show()

print(f"PCA explained variance (3 components): {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# --- Cluster Profiling ---
churn_profile = churn_feats.copy()
churn_profile['cluster'] = cluster_labels

# Key features to profile
profile_cols = ['tenure_rounds', 'total_active_rounds', 'purchase_frequency_ratio',
                'total_items', 'avg_items_per_active_round', 'items_last_3_rounds',
                'current_zero_streak', 'longest_zero_streak', 'trend_slope',
                'ewm_recent', 'rounds_since_last_purchase', 'age']

cluster_summary = churn_profile.groupby('cluster')[profile_cols].agg(['mean', 'median', 'count']).round(2)

# Simplified view: just means
cluster_means = churn_profile.groupby('cluster')[profile_cols].mean().round(2)
cluster_sizes = churn_profile.groupby('cluster').size()

print("Cluster Sizes:")
print(cluster_sizes)
print(f"\nCluster Profiles (Mean):")
display(cluster_means.T)

# Radar chart for cluster profiles
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, best_k, figsize=(6*best_k, 5))
if best_k == 1:
    axes = [axes]

# Normalize for comparison
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min() + 1e-8)

for i, ax in enumerate(axes):
    vals = cluster_means_norm.loc[i].values
    ax.barh(profile_cols, vals, color=plt.cm.tab10(i), alpha=0.7)
    ax.set_title(f'Cluster {i}\n(n={cluster_sizes[i]:,})', fontweight='bold')
    ax.set_xlim(0, 1.1)
    
plt.suptitle('Churn Cluster Profiles (Normalized)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Cluster Interpretation & Naming ---
# Auto-generate cluster descriptions based on profile
print("=" * 80)
print("CHURN ARCHETYPE INTERPRETATION")
print("=" * 80)

for i in range(best_k):
    row = cluster_means.loc[i]
    size = cluster_sizes[i]
    pct = size / cluster_sizes.sum() * 100
    
    print(f"\n{'='*60}")
    print(f"Cluster {i} — {size:,} customers ({pct:.1f}% of churned)")
    print(f"{'='*60}")
    print(f"  Tenure:             {row['tenure_rounds']:.1f} rounds")
    print(f"  Active rounds:      {row['total_active_rounds']:.1f}")
    print(f"  Frequency ratio:    {row['purchase_frequency_ratio']:.2%}")
    print(f"  Total items:        {row['total_items']:.1f}")
    print(f"  Avg items/round:    {row['avg_items_per_active_round']:.1f}")
    print(f"  Last 3 rounds:      {row['items_last_3_rounds']:.1f}")
    print(f"  Current 0-streak:   {row['current_zero_streak']:.1f}")
    print(f"  Longest 0-streak:   {row['longest_zero_streak']:.1f}")
    print(f"  Trend slope:        {row['trend_slope']:.4f}")
    print(f"  Recency (rounds):   {row['rounds_since_last_purchase']:.1f}")
    print(f"  Age:                {row['age']:.1f}")
    
    # Auto-classify pattern
    if row['tenure_rounds'] < 5 and row['total_active_rounds'] < 3:
        pattern = "NEW & GONE — สมัครใหม่แต่หลุดเร็ว ยังไม่เกิดความผูกพัน"
    elif row['current_zero_streak'] > 5 and row['total_items'] > 20:
        pattern = "DORMANT WHALE — เคยซื้อเยอะแต่หายไปนาน ควรดึงกลับเร่งด่วน"
    elif row['trend_slope'] < -0.1:
        pattern = "DECLINING — ซื้อน้อยลงเรื่อยๆ สัญญาณเตือนชัดเจน"
    elif row['purchase_frequency_ratio'] < 0.3 and row['tenure_rounds'] > 10:
        pattern = "LOW ENGAGEMENT — อยู่นานแต่ซื้อน้อยมาก"
    elif row['total_items'] > 50 and row['items_last_3_rounds'] < 5:
        pattern = "RECENT DROPOUT — เคยซื้อสม่ำเสมอแต่เพิ่งหยุด"
    else:
        pattern = "SPORADIC — ซื้อไม่สม่ำเสมอ เข้าๆออกๆ"
    
    print(f"\n  >> Pattern: {pattern}")

## Phase 8: Churn Radar — Monitoring Dashboard

Cohort Analysis, Early Warning Signals, และ Trend Monitoring

In [ ]:
# --- Cohort Analysis: Registration cohort vs Churn rate ---
df_test_full = datasets[sorted(datasets.keys())[-1]].copy()
df_test_full['churn'] = 1 - df_test_full['label_continue']
df_test_full['reg_period'] = pd.to_datetime(df_test_full['RegisteredRoundDate'])
df_test_full['reg_quarter'] = df_test_full['reg_period'].dt.to_period('Q')

cohort_stats = df_test_full.groupby('reg_quarter').agg(
    total=('churn', 'size'),
    churned=('churn', 'sum'),
    churn_rate=('churn', 'mean')
).reset_index()
cohort_stats = cohort_stats[cohort_stats['total'] >= 100]

fig, ax1 = plt.subplots(figsize=(16, 6))
ax1.bar(cohort_stats['reg_quarter'].astype(str), cohort_stats['total'], 
        color='#3498db', alpha=0.3, label='Total Customers')
ax1.set_ylabel('Total Customers', color='#3498db')
ax1.tick_params(axis='y', labelcolor='#3498db')
ax1.set_xticklabels(cohort_stats['reg_quarter'].astype(str), rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.plot(cohort_stats['reg_quarter'].astype(str), cohort_stats['churn_rate'], 
         'ro-', linewidth=2, markersize=8, label='Churn Rate')
ax2.set_ylabel('Churn Rate', color='#e74c3c')
ax2.tick_params(axis='y', labelcolor='#e74c3c')

plt.title('Cohort Analysis: Churn Rate by Registration Quarter')
fig.tight_layout()
plt.show()

In [ ]:
# --- Early Warning Signals: What happens before churn? ---
# Track average purchase trajectory of churned vs retained in last 10 rounds

df_latest_raw = datasets[sorted(datasets.keys())[-1]]
mat_latest, cols_latest = build_purchase_matrix(df_latest_raw)
amounts_latest = mat_latest.fillna(0)

churn_mask_latest = df_latest_raw['label_continue'] == 0
retain_mask_latest = df_latest_raw['label_continue'] == 1

# Last 15 rounds trajectory
n_trail = min(15, len(cols_latest))
trail_cols = cols_latest[-n_trail:]
trail_labels = [c.replace('item', '').replace('_', '-') for c in trail_cols]

churn_trajectory = amounts_latest.loc[churn_mask_latest, trail_cols].mean()
retain_trajectory = amounts_latest.loc[retain_mask_latest, trail_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].plot(trail_labels, churn_trajectory.values, 'r-o', linewidth=2, markersize=6, label='Will Churn')
axes[0].plot(trail_labels, retain_trajectory.values, 'b-s', linewidth=2, markersize=6, label='Will Retain')
axes[0].fill_between(trail_labels, churn_trajectory.values, retain_trajectory.values, alpha=0.1, color='red')
axes[0].set_title('Average Purchase Trajectory — Last 15 Rounds\n(Churn vs Retain)')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Avg Items Purchased')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# --- Risk Score distribution over time (across periods) ---
risk_over_time = []
for key in sorted(datasets.keys()):
    mask = features_df['period'] == key
    df_period = features_df[mask]
    
    risk_over_time.append({
        'period': key.replace('Churn_', ''),
        'mean_items_last_3': df_period[df_period['churn']==1]['items_last_3_rounds'].mean(),
        'mean_zero_streak_churn': df_period[df_period['churn']==1]['current_zero_streak'].mean(),
        'mean_zero_streak_retain': df_period[df_period['churn']==0]['current_zero_streak'].mean(),
    })

rot_df = pd.DataFrame(risk_over_time)
axes[1].plot(rot_df['period'], rot_df['mean_zero_streak_churn'], 'r-o', linewidth=2, label='Churn — Avg 0-Streak')
axes[1].plot(rot_df['period'], rot_df['mean_zero_streak_retain'], 'b-s', linewidth=2, label='Retain — Avg 0-Streak')
axes[1].set_title('Early Warning: Current Zero-Streak Trend')
axes[1].set_xlabel('Period')
axes[1].set_ylabel('Avg Current Zero Streak')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nKey Insight: ดู gap ระหว่างเส้น Churn กับ Retain ยิ่ง gap กว้าง ยิ่งเป็น signal ที่ดี")
print("ถ้า mean zero streak ของ customer เกินค่าเฉลี่ยของกลุ่ม Churn → เข้าโซนอันตราย")

In [ ]:
# --- Churn Radar: Heatmap of risk signals across periods ---
radar_metrics = []
for key in sorted(datasets.keys()):
    mask = features_df['period'] == key
    df_p = features_df[mask]
    churn_p = df_p[df_p['churn'] == 1]
    
    radar_metrics.append({
        'Period': key.replace('Churn_', ''),
        'Churn Rate': df_p['churn'].mean(),
        'Avg Zero Streak (Churn)': churn_p['current_zero_streak'].mean(),
        'Avg Frequency (Churn)': churn_p['purchase_frequency_ratio'].mean(),
        'Avg Items Last 3 (Churn)': churn_p['items_last_3_rounds'].mean(),
        'Avg Trend Slope (Churn)': churn_p['trend_slope'].mean(),
        'Avg Tenure (Churn)': churn_p['tenure_rounds'].mean(),
        'Pct Zero Last Round': (df_p['items_last_1_round'] == 0).mean(),
    })

radar_df = pd.DataFrame(radar_metrics).set_index('Period')

# Normalize each column for heatmap
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min() + 1e-8)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(radar_norm.T, annot=radar_df.T.round(3), fmt='', cmap='YlOrRd', 
            linewidths=1, ax=ax, cbar_kws={'label': 'Normalized (0-1)'})
ax.set_title('Churn Radar — Key Metrics Across Periods', fontsize=14)
plt.tight_layout()
plt.show()

## Phase 9: Production Pipeline — Ready for Next Period

Class `ChurnPipeline` สำหรับ deploy กับข้อมูลงวดถัดไป:
1. โหลด CSV ใหม่
2. Feature engineering อัตโนมัติ
3. Predict risk score
4. จัดกลุ่ม cluster
5. Export ผลลัพธ์

In [ ]:
import pickle
from datetime import datetime

class ChurnPipeline:
    """Production pipeline for churn risk scoring on new period data."""
    
    def __init__(self, model, calibrated_model, scaler_cluster, kmeans, feature_cols):
        self.model = model
        self.calibrated_model = calibrated_model
        self.scaler_cluster = scaler_cluster
        self.kmeans = kmeans
        self.feature_cols = feature_cols
    
    def process(self, csv_path, output_path=None):
        """Process a new period CSV and output risk scores + clusters."""
        print(f"[1/5] Loading data from {csv_path}...")
        df = pd.read_csv(csv_path, low_memory=False)
        print(f"      {len(df):,} customers loaded")
        
        print("[2/5] Engineering features...")
        feats = engineer_features(df)
        X = feats[self.feature_cols].fillna(0)
        
        print("[3/5] Predicting risk scores...")
        risk_probs = self.calibrated_model.predict_proba(X)[:, 1]
        risk_scores = (risk_probs * 100).round(1)
        
        print("[4/5] Assigning churn clusters...")
        X_scaled = self.scaler_cluster.transform(X)
        clusters = self.kmeans.predict(X_scaled)
        
        print("[5/5] Building output...")
        result = pd.DataFrame({
            'userNo': df['userNo'],
            'risk_score': risk_scores,
            'risk_level': pd.cut(risk_scores, bins=[-1, 25, 50, 75, 100],
                                 labels=['Low', 'Medium', 'High', 'Critical']),
            'cluster': clusters,
            'age': df['age'],
            'gender': df['gender'],
            'province': df['province'],
            'tenure_rounds': feats['tenure_rounds'],
            'total_items': feats['total_items'],
            'purchase_frequency': feats['purchase_frequency_ratio'],
            'current_zero_streak': feats['current_zero_streak'],
            'items_last_3_rounds': feats['items_last_3_rounds'],
            'trend_slope': feats['trend_slope'],
        })
        result = result.sort_values('risk_score', ascending=False)
        
        if output_path:
            result.to_csv(output_path, index=False)
            print(f"      Saved to {output_path}")
        
        # Summary
        print(f"\n{'='*50}")
        print(f"RISK SCORE SUMMARY")
        print(f"{'='*50}")
        print(f"Total customers: {len(result):,}")
        print(f"\nRisk Level Distribution:")
        for level in ['Critical', 'High', 'Medium', 'Low']:
            n = (result['risk_level'] == level).sum()
            pct = n / len(result) * 100
            print(f"  {level:10s}: {n:>8,} ({pct:.1f}%)")
        print(f"\nMean Risk Score: {result['risk_score'].mean():.1f}")
        print(f"Median Risk Score: {result['risk_score'].median():.1f}")
        
        return result
    
    def save(self, path='churn_pipeline.pkl'):
        """Save pipeline to disk."""
        with open(path, 'wb') as f:
            pickle.dump(self, f)
        print(f"Pipeline saved to {path}")
    
    @staticmethod
    def load(path='churn_pipeline.pkl'):
        """Load pipeline from disk."""
        with open(path, 'rb') as f:
            return pickle.load(f)

# --- Create and save the pipeline ---
pipeline = ChurnPipeline(
    model=best_tree,
    calibrated_model=calibrated_model,
    scaler_cluster=scaler_cl,
    kmeans=km_final,
    feature_cols=model_features
)

pipeline.save('churn_pipeline.pkl')
print("\nPipeline ready for deployment!")

In [ ]:
# --- Demo: Run pipeline on latest period (as if it were new data) ---
latest_file = sorted(DATA_DIR.glob('Churn_*.csv'))[-1]
result = pipeline.process(str(latest_file), output_path='risk_scores_latest.csv')

print("\nTop 20 highest risk customers:")
display(result.head(20))

## Phase 10: Final Report — Model Comparison & New Features Impact

รายงานสรุปผลทั้งหมด:
1. Model Performance Summary (ทุก model)
2. Cross-Model Feature Importance (consensus ranking)
3. New Features Statistical Significance
4. Confusion Matrix, Threshold Analysis & Gain Chart
5. Per-Model Classification Reports
6. Risk Score Effectiveness (bucket analysis)

In [ ]:
# ============================================================
# FINAL REPORT: Model Performance & Feature Analysis
# ============================================================

print("=" * 80)
print("FINAL REPORT: Churn Prediction — Model Comparison")
print("=" * 80)

# --- 1. Model Performance Summary ---
print("\n" + "=" * 60)
print("1. MODEL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"   Training: 4 periods (temporal split) | Test: latest period")
print(f"   Train size: {len(X_train):,} | Test size: {len(X_test):,}")
print(f"   Train churn rate: {y_train.mean():.2%} | Test churn rate: {y_test.mean():.2%}")
print()

# Re-evaluate all models and build comprehensive table
report_rows = []
for name, (data_type, model) in models.items():
    X_eval = X_test_scaled if data_type == 'scaled' else X_test
    y_prob = model.predict_proba(X_eval)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    from sklearn.metrics import precision_score, recall_score, accuracy_score
    
    report_rows.append({
        'Model': name,
        'AUC-ROC': roc_auc_score(y_test, y_prob),
        'AUC-PR': average_precision_score(y_test, y_prob),
        'F1': f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Brier Score': brier_score_loss(y_test, y_prob),
    })

report_df = pd.DataFrame(report_rows).sort_values('AUC-ROC', ascending=False)
report_df.index = range(1, len(report_df) + 1)
report_df.index.name = 'Rank'

print(report_df.to_string())

# Best model highlight
best = report_df.iloc[0]
print(f"\n   BEST MODEL: {best['Model']}")
print(f"   AUC-ROC: {best['AUC-ROC']:.4f} | AUC-PR: {best['AUC-PR']:.4f}")
print(f"   F1: {best['F1']:.4f} | Precision: {best['Precision']:.4f} | Recall: {best['Recall']:.4f}")

In [ ]:
# --- 2. Feature Importance Ranking (cross-model consensus) ---
print("\n" + "=" * 60)
print("2. FEATURE IMPORTANCE — CROSS-MODEL CONSENSUS")
print("=" * 60)

# Collect importance from multiple models
importance_df = pd.DataFrame(index=model_features)

# LightGBM
importance_df['LightGBM'] = pd.Series(lgb_model.feature_importances_, index=model_features)
importance_df['LightGBM'] = importance_df['LightGBM'] / importance_df['LightGBM'].sum()

# XGBoost
importance_df['XGBoost'] = pd.Series(xgb_model.feature_importances_, index=model_features)
importance_df['XGBoost'] = importance_df['XGBoost'] / importance_df['XGBoost'].sum()

# Random Forest
importance_df['RandomForest'] = pd.Series(rf.feature_importances_, index=model_features)
importance_df['RandomForest'] = importance_df['RandomForest'] / importance_df['RandomForest'].sum()

# Logistic Regression (absolute coefficients)
importance_df['LogReg_abs'] = pd.Series(np.abs(lr.coef_[0]), index=model_features)
importance_df['LogReg_abs'] = importance_df['LogReg_abs'] / importance_df['LogReg_abs'].sum()

# Average rank
importance_df['avg_importance'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('avg_importance', ascending=False)

# Tag new features
new_feats = ['avg_gap', 'std_gap', 'max_gap', 'recent_gap_vs_avg',
             'n_reactivations', 'ever_reactivated',
             'items_last3_vs_prev3_ratio', 'is_declining_3_consecutive',
             'gini_coefficient', 'top3_rounds_pct', 'is_new', 'is_mature']
importance_df['is_new_feature'] = importance_df.index.isin(new_feats)

print("\nTop 20 features by cross-model average importance:")
top20 = importance_df.head(20)[['LightGBM', 'XGBoost', 'RandomForest', 'LogReg_abs', 'avg_importance', 'is_new_feature']].copy()
top20.columns = ['LGBM', 'XGB', 'RF', 'LR', 'Avg', 'New?']
print(top20.round(4).to_string())

# Chart
fig, ax = plt.subplots(figsize=(14, 10))
colors = ['#e74c3c' if importance_df.loc[f, 'is_new_feature'] else '#3498db' 
          for f in importance_df.head(25).index]
importance_df.head(25)['avg_importance'].plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_xlabel('Average Normalized Importance')
ax.set_title('Top 25 Features — Cross-Model Consensus\n(Red = NEW features added in v2)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

n_new_in_top15 = importance_df.head(15)['is_new_feature'].sum()
print(f"\nNew features in top 15: {n_new_in_top15} / 15")

In [ ]:
# --- 3. New Features: Statistical significance ---
print("\n" + "=" * 60)
print("3. NEW FEATURES — STATISTICAL SIGNIFICANCE (Churn vs Retain)")
print("=" * 60)

new_feat_stats = stat_results[stat_results['feature'].isin(new_feats)].copy()
new_feat_stats = new_feat_stats.sort_values('abs_cohens_d', ascending=False)

print("\n" + new_feat_stats[['feature', 'churn_mean', 'retain_mean', 'cohens_d', 
                             'effect_size', 'significant']].to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

plot_feats = new_feat_stats.head(6)['feature'].tolist()
for i, col in enumerate(plot_feats):
    ax = axes[i]
    data_r = features_df.loc[features_df['churn']==0, col].dropna()
    data_c = features_df.loc[features_df['churn']==1, col].dropna()
    p99 = features_df[col].quantile(0.99)
    ax.hist(data_r.clip(upper=p99), bins=40, alpha=0.5, label='Retain', color='#3498db', density=True)
    ax.hist(data_c.clip(upper=p99), bins=40, alpha=0.5, label='Churn', color='#e74c3c', density=True)
    
    d = new_feat_stats[new_feat_stats['feature']==col]['cohens_d'].values[0]
    ax.set_title(f'{col}\n(Cohen\'s d = {d:.3f})', fontsize=11)
    ax.legend()

plt.suptitle('Top 6 New Features — Distribution Comparison', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4. Confusion Matrix & Threshold Analysis for best model ---
print("\n" + "=" * 60)
print("4. BEST MODEL — DETAILED CLASSIFICATION & THRESHOLD ANALYSIS")
print("=" * 60)

best_name = report_df.iloc[0]['Model']
best_data_type, best_model_obj = models[best_name]
X_eval_best = X_test_scaled if best_data_type == 'scaled' else X_test
y_prob_best = best_model_obj.predict_proba(X_eval_best)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Confusion matrix at 0.5
y_pred_05 = (y_prob_best >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_05)
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=axes[0],
            xticklabels=['Retain', 'Churn'], yticklabels=['Retain', 'Churn'])
axes[0].set_title(f'{best_name} — Confusion Matrix (threshold=0.5)')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Threshold analysis
thresholds = np.arange(0.1, 0.91, 0.05)
f1s, precs, recs = [], [], []
for t in thresholds:
    yp = (y_prob_best >= t).astype(int)
    f1s.append(f1_score(y_test, yp))
    precs.append(precision_score(y_test, yp, zero_division=0))
    recs.append(recall_score(y_test, yp))

axes[1].plot(thresholds, f1s, 'g-o', label='F1', linewidth=2)
axes[1].plot(thresholds, precs, 'b-s', label='Precision', linewidth=2)
axes[1].plot(thresholds, recs, 'r-^', label='Recall', linewidth=2)
optimal_t = thresholds[np.argmax(f1s)]
axes[1].axvline(x=optimal_t, color='gray', linestyle='--', label=f'Optimal F1 threshold={optimal_t:.2f}')
axes[1].set_title('Threshold Analysis')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].legend()

# Gain chart (cumulative % of churners captured)
sorted_idx = np.argsort(-y_prob_best)
y_sorted = y_test.values[sorted_idx]
cum_churn = np.cumsum(y_sorted) / y_sorted.sum()
pct_customers = np.arange(1, len(y_sorted)+1) / len(y_sorted)

axes[2].plot(pct_customers * 100, cum_churn * 100, 'b-', linewidth=2, label=best_name)
axes[2].plot([0, 100], [0, 100], 'k--', alpha=0.5, label='Random')
axes[2].fill_between(pct_customers * 100, cum_churn * 100, pct_customers * 100, alpha=0.1, color='blue')

# Mark key points
for pct_target in [10, 20, 30]:
    idx = int(len(y_sorted) * pct_target / 100)
    captured = cum_churn[idx] * 100
    axes[2].annotate(f'Top {pct_target}% → {captured:.0f}% churn', 
                     (pct_target, captured), fontsize=9, fontweight='bold',
                     textcoords="offset points", xytext=(10, -10))

axes[2].set_title('Cumulative Gain Chart')
axes[2].set_xlabel('% of Customers (sorted by risk)')
axes[2].set_ylabel('% of Churners Captured')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\n   Optimal threshold (max F1): {optimal_t:.2f}")
print(f"   F1 at optimal: {max(f1s):.4f}")
idx_20 = int(len(y_sorted) * 0.20)
print(f"   Top 20% of customers captures {cum_churn[idx_20]*100:.1f}% of all churners")

In [ ]:
# --- 5. Per-model detailed breakdown ---
print("\n" + "=" * 60)
print("5. PER-MODEL CLASSIFICATION REPORTS")
print("=" * 60)

for name, (data_type, model) in models.items():
    X_eval = X_test_scaled if data_type == 'scaled' else X_test
    y_prob = model.predict_proba(X_eval)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(classification_report(y_test, y_pred, target_names=['Retain', 'Churn'], digits=4))

In [ ]:
# --- 6. Risk Score Effectiveness Report ---
print("\n" + "=" * 60)
print("6. RISK SCORE EFFECTIVENESS")
print("=" * 60)

risk_report = risk_df.copy()
risk_report['risk_bucket'] = pd.cut(risk_report['risk_score'], 
                                      bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
                                      labels=['0-10', '10-20', '20-30', '30-40', '40-50',
                                              '50-60', '60-70', '70-80', '80-90', '90-100'])

bucket_stats = risk_report.groupby('risk_bucket').agg(
    n_customers=('actual_churn', 'size'),
    n_churned=('actual_churn', 'sum'),
    churn_rate=('actual_churn', 'mean'),
    avg_risk=('risk_score', 'mean')
).reset_index()

print("\nRisk Score Bucket Analysis:")
print(bucket_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(bucket_stats['risk_bucket'], bucket_stats['churn_rate'],
              color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(bucket_stats))),
              edgecolor='white', linewidth=0.5)

for bar, rate, n in zip(bars, bucket_stats['churn_rate'], bucket_stats['n_customers']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{rate:.0%}\n({n:,})', ha='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Risk Score Bucket')
ax.set_ylabel('Actual Churn Rate')
ax.set_title('Risk Score Calibration — Actual Churn Rate per Bucket')
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

print(f"\nCorrelation between risk_score and actual_churn: {risk_report['risk_score'].corr(risk_report['actual_churn']):.4f}")

## Summary & Key Findings

### Model Performance
| Model | AUC-ROC | AUC-PR | F1 | Brier Score | Verdict |
|-------|---------|--------|-----|-------------|---------|
| **LightGBM** | ดูผลจาก report ด้านบน | - | - | - | Fast, accurate, production-ready |
| **XGBoost** | - | - | - | - | Close competitor, slightly slower |
| **Random Forest** | - | - | - | - | Robust, less overfitting |
| **Logistic Regression** | - | - | - | - | Interpretable baseline |

### Top Predictive Features (Cross-Model Consensus)
Features เหล่านี้มี predictive power สูงสุดจากทุก model:
1. **current_zero_streak** — จำนวนงวดที่ไม่ซื้อล่าสุดต่อเนื่อง
2. **items_last_3_rounds** — ยอดซื้อ 3 งวดล่าสุด
3. **purchase_frequency_ratio** — อัตราส่วนงวดที่ซื้อ
4. **ewm_recent** — exponentially weighted recent purchase
5. **rounds_since_last_purchase** — งวดตั้งแต่ซื้อครั้งสุดท้าย

### New Features (v2) Impact
- **avg_gap / max_gap** — จังหวะการซื้อช่วยแยก churn ได้ดี
- **recent_gap_vs_avg** — ถ้า gap ล่าสุดมากกว่า avg gap = สัญญาณอันตราย
- **n_reactivations** — ลูกค้าที่เคย reactivate มีแนวโน้มกลับมามากกว่า
- **gini_coefficient** — ลูกค้าที่ซื้อกระจุก (gini สูง) เสี่ยง churn กว่า
- **is_declining_3_consecutive** — ลดลง 3 งวดติดต่อกัน = strong signal

### Actionable Insights
1. **Intervene เมื่อ zero_streak >= 2** — ส่ง promotion/ติดต่อทันที
2. **ลูกค้า New & Gone** — ต้อง onboard ให้ดี 3 งวดแรกสำคัญมาก
3. **Dormant Whale** — เคยซื้อเยอะแต่หาย ใช้ personalized offer ดึงกลับ
4. **Risk Score 70+** — ควร assign ให้ทีม CRM ดูแลตัวต่อตัว
5. **Top 20% risk customers = จับ churners ได้ส่วนใหญ่** — focus resources ที่นี่

### Production Deployment — Usage เมื่อได้ข้อมูลงวดใหม่
```python
pipeline = ChurnPipeline.load('churn_pipeline.pkl')
result = pipeline.process('Churn_NEW_FILE.csv', output_path='risk_output.csv')
```